In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    trim,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()

In [0]:
def load_curated_entity_relationships_arotron(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "EntityRelationshipId    string,"
        #     "SourceSystem  string,"
        #     "PortfolioKey  string,"
        #     "PortfolioName string,"
        #     "IngestionTimestamp   timestamp,"
        #     "BatchId  string,"
        #     "EntityRelationshipKey   string,"
        #     "EntityKey    string,"
        #     "RelatedEntityKey string,"
        #     "EntityRelationshipType    string,"
        #     "EntityRelationshipSubType    string,"
        #     "EntityType    string,"
        #     "RelatedEntityType    string,"
        #     "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.EntityRelationships"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "arotron",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = ["PortfolioName", "Prefix", "unit_code_profits"]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )


        # ================== eaiSIG_BE ==================
        eaiSIG_BE_extra_filters = None

        if eaiSIG_BE_extra_filters is not None:
            eaiSIG_BE_extra_filters.append(date_filter)
            eaiSIG_BE_final_filters = " AND ".join(
                eaiSIG_BE_extra_filters
            )
        else:
            eaiSIG_BE_final_filters = date_filter

        eaiSIG_BE_selected_fields = [
            "eaiSIG_BE_CASE_MASTER_ACC",
            "eaiSIG_BE_CASE_S_ID",
            "eaiSIG_BE_CUST_S_UNID",
            "businessdate",
        ]

        eaiSIG_BE = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_eaiSIG_BE",
            eaiSIG_BE_final_filters,
            eaiSIG_BE_selected_fields,
        )

        # =============== Join with portfolio ===============
        eaiSIG_BE = eaiSIG_BE.join(
            portfolio,
            on=substring(eaiSIG_BE["eaiSIG_BE_CUST_S_UNID"], 1, 3) == portfolio["Prefix"],
            how="left"
        )

        result_df = eaiSIG_BE.filter(
                    (eaiSIG_BE.eaiSIG_BE_CASE_MASTER_ACC.cast("bigint") > 0)
                ).distinct()

        # HashKeys
        result_df_primary_key_fields = [
            eaiSIG_BE.eaiSIG_BE_CASE_MASTER_ACC,
            eaiSIG_BE.eaiSIG_BE_CASE_S_ID,
            lit("Master - Slave Rltnp"),
            lit("arotron"),
        ]

        result_df = result_df.withColumn(
            "EntityRelationshipId",
            sha2(concat_ws("", *result_df_primary_key_fields), 256),
        ).withColumn(
            "EntityRelationshipKey",
            concat_ws(
                ";",
                eaiSIG_BE.eaiSIG_BE_CASE_MASTER_ACC,
                eaiSIG_BE.eaiSIG_BE_CASE_S_ID,
                lit("Master - Slave Rltnp"),
            ),
        )

        # ================== Selection of Final Columns ==================
        result_df_arotron = result_df.select(
            result_df.EntityRelationshipId.alias("EntityRelationshipId"),
            lit("arotron").alias("SourceSystem"),
            coalesce(portfolio.unit_code_profits, lit("-9998")).cast("string").alias("PortfolioKey"),
            coalesce(portfolio.PortfolioName, lit("Core Wholesale")).cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            result_df.EntityRelationshipKey.alias("EntityRelationshipKey"),
            trim(result_df.eaiSIG_BE_CASE_MASTER_ACC).alias("EntityKey"),
            trim(result_df.eaiSIG_BE_CASE_S_ID).alias("RelatedEntityKey"),
            lit("Master - Slave Rltnp").alias("EntityRelationshipType"),
            lit(None).alias("EntityRelationshipSubType"),
            lit("Exposure").alias("EntityType"),
            lit("Exposure").alias("RelatedEntityType"),
            result_df.businessdate.alias("BusinessDate")
        )

        # ================== Append ==================
        result_df_arotron = (
            result_df_arotron.distinct()
        )  # Eliminate duplicates in SourceSystem
        result_df_arotron.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")

In [0]:
load_curated_entity_relationships_arotron(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)